In [2]:
### uniform grid
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from scipy.ndimage import gaussian_filter
import gaussian_random_fields as gr
from time import time
from MT2D_secondary_direct import *
from timeit import default_timer
from scipy.interpolate import interp2d
import scipy.io as scio
import scipy.sparse as scipa 
import scipy.sparse.linalg as scilg
import cmath as cm
from scipy.interpolate import RegularGridInterpolator
import random
import sys
import os
import ray as ray

# Set parameters (for generating training/test data for MT forward modeling, corresponding to data generation parameters in the paper)
np_dtype = np.float16 # Numerical precision (use double-precision floating point to ensure calculation accuracy, especially for complex electromagnetic field calculations)
n_sample = 10       # Number of samples to generate (i.e., number of conductivity models for model training and validation)
num_cpus=1      # Number of CPUs used for computation (multi-process parallel data generation to accelerate large-scale sample generation)
num_gpus= 1
n_freq = 16            # Number of frequencies (corresponds to multi-frequency settings in the paper, verifying cross-frequency generalization of the model)
nza = 10               # Number of air layer grids (grids above the surface, extremely low conductivity to simulate air medium)
size_b = 10            # Number of grids in the boundary region (edge grids of the computational domain for handling boundary conditions)
size_k = 43            # Number of grids in the core region (main subsurface simulation area, where conductivity variations primarily occur)
alpha_l = [4.0,5.0,6.0,7.0,8.0]  # Length scale parameters for Gaussian Random Field (GRF) (controls smoothness of conductivity structures; different values generate diverse geological structures)
if not ray.is_initialized():
    ray.init(num_cpus=num_cpus, num_gpus=num_gpus,ignore_reinit_error=True)

2026-05-13 19:52:02,788	INFO worker.py:1927 -- Started a local Ray instance.


In [3]:
def generate_model(alpha_l, n_sample, n_freq, nza, size_b, size_k):
    """
    Generate conductivity models and grid parameters for Magnetotelluric (MT) forward modeling
    Function: Construct a non-uniform grid including air layer, core region and boundary regions, 
              generate geologically consistent random conductivity structures using Gaussian Random Field (GRF), 
              and process boundaries to ensure physical rationality
    Parameters:
        alpha_l: Length scale of Gaussian Random Field (controls structure smoothness)
        n_sample: Number of model samples to generate
        n_freq: Number of frequencies
        nza: Number of air layer grids
        size_b: Number of boundary region grids
        size_k: Number of core region grids
    Returns:
        zn/yn: Complete grid coordinates in z/y directions; freq: Frequency array; ry: Receiver positions;
        model: Complete conductivity model; zn0/yn0: Core region grids; model_k: Core region conductivity
    """
    # Depth and horizontal range of the core region (unit: meters)
    z = 75e3  # Core region depth (75 kilometers)
    y = 75e3  # Core region horizontal range (-75 km to 75 km)
    
    # Expansion multiples for boundary regions (non-uniform grid: sparse at boundaries, dense in core region)
    multiple_t = 1.5  # Top (air layer) expansion multiple
    multiple_b = 2.0  # Bottom boundary expansion multiple
    multiple_l = 2.5  # Left boundary expansion multiple
    multiple_r = 2.5  # Right boundary expansion multiple
    
    # Generate frequency array (logarithmic distribution from 1Hz to 0.01Hz, n_freq points in total, matching common MT frequency range)
    freq = np.logspace(np.log10(1), np.log10(1/100), n_freq)

    # Construct z-direction grid (vertical: air layer + core region + bottom boundary)
    # Air layer grid (above surface, logarithmic distribution for dense grids near the surface)
    z_air = -(np.logspace(np.log10(50e3), np.log10(50e3 + multiple_t * z), nza + 1) - 50e3)[::-1]
    # Core region z-direction grid (uniform distribution)
    zn0 = np.linspace(0, z, size_k + 1)
    # Bottom boundary grid (logarithmic distribution, sparse away from core region)
    z_b = np.logspace(np.log10(zn0[-1]), np.log10(multiple_b * zn0[-1]), size_b + 1)
    # Concatenate complete z-direction grid (air layer + core + bottom boundary)
    zn = np.concatenate((z_air[:-1], zn0, z_b[1:]))

    # Construct y-direction grid (horizontal: left boundary + core region + right boundary)
    # Core region y-direction grid (uniform distribution from -y to y)
    yn0 = np.linspace(-y, y, size_k + 1)
    # Left boundary grid (logarithmic distribution, sparse away from core region)
    y_l = -(np.logspace(np.log10(multiple_l * yn0[-1]), np.log10(yn0[-1]), size_b + 1))
    # Right boundary grid (logarithmic distribution, sparse away from core region)
    y_r = np.logspace(np.log10(yn0[-1]), np.log10(multiple_r * yn0[-1]), size_b + 1)
    # Concatenate complete y-direction grid (left boundary + core + right boundary)
    yn = np.concatenate((y_l[:-1], yn0, y_r[1:]))
    
    # Receiver positions (use horizontal grids of the core region, corresponding to surface observation points)
    ry = yn0

    # Calculate dimensions of the complete grid
    len_z = nza + size_b + size_k  # Total z-direction grids (air + core + bottom boundary)
    len_y = 2 * size_b + size_k    # Total y-direction grids (left boundary + core + right boundary)
    # Initialize conductivity model (log10(conductivity), initial value -2 corresponds to 10^-2 S/m, close to average earth conductivity)
    model = np.ones((n_sample, len_z, len_y)) * (-2)

    # Conductivity range for Gaussian Random Field (log10 values): 
    # sig_up=-4 (10^-4 S/m, high resistivity), sig_down=0 (1 S/m, low resistivity)
    sig_up, sig_down = -4, 0  

    # Generate conductivity model for each sample
    for ii in range(n_sample):
        model0 = 0.0  # Initialize core region conductivity
        
        # Superpose Gaussian Random Fields (GRF) with different length scales to generate diverse geological structures
        for alpha in alpha_l:
            # Generate single GRF (alpha controls smoothness: larger alpha = smoother structure)
            model_temp0 = gr.gaussian_random_field(
                alpha=alpha, size=size_k, mode='bound', set_1=sig_up, set_2=sig_down
            )
            model0 += model_temp0  # Superpose multiple GRFs
        
        # Smooth and normalize the generated conductivity field to a reasonable range
        min0 = np.min(model_temp0)  # Min value of raw GRF
        max0 = np.max(model_temp0)  # Max value of raw GRF
        model0 = gaussian_filter(model0, sigma=2) / len(alpha_l)  # Gaussian smoothing and averaging
        min1 = np.min(model0)  # Min value after smoothing
        max1 = np.max(model0)  # Max value after smoothing
        # Remap to the original GRF range to ensure conductivity variation amplitude
        model0 = (model0 - min1) * ((max0 - min0) / (max1 - min1)) + min0
        
        # Assign core region conductivity to the complete model (exclude air layer and boundary regions)
        model[ii, nza:-size_b, size_b:-size_b] = model0
        
        # Set random conductivity for the top of the core region (near surface: -3 to -1 corresponds to 10^-3 to 10^-1 S/m)
        sig_0 = np.random.uniform(-3, -1, 1)[0]
        model[ii, nza:nza + 1, size_b:-size_b] = sig_0 
        
        # Initialize left/right boundary conductivity (-2 corresponds to 10^-2 S/m)
        model[ii, nza:, :size_b] = -2  # Left boundary
        model[ii, nza:, -size_b:] = -2  # Right boundary

        # Left boundary interpolation (ensure smooth transition from core to boundary)
        # Construct interpolation source data (core left edge + boundary value)
        sig_left = np.concatenate((model[ii, :, size_b:size_b + 1],
                                   (-2) * np.ones_like(model[ii, :, size_b:size_b + 1])), 1).T
        # Define linear interpolation function
        f_left = RegularGridInterpolator(
            (np.concatenate([[yn[int(size_b)]], [yn[int(2 * size_b / 3)]]]), zn[:-1]),
            sig_left, method='linear', bounds_error=False, fill_value=None
        )
        # Interpolate and fill left boundary grids
        for iy in range(int(2 * size_b / 3), size_b):
            model[ii, :, iy] = f_left((np.full_like(zn[:-1], yn[iy]), zn[:-1]))

        # Right boundary interpolation (same logic for smooth transition)
        sig_right = np.concatenate((model[ii, :, -size_b - 1:-size_b],
                                    (-2) * np.ones_like(model[ii, :, -size_b - 1:-size_b])), 1).T
        f_right = RegularGridInterpolator(
            (np.concatenate([yn[-size_b - 1:-size_b], [yn[int(-2 * size_b / 3)]]]), zn[:-1]),
            sig_right, method='linear', bounds_error=False, fill_value=None
        )
        for iy in range(-size_b, int(-2 * size_b / 3), 1):
            model[ii, :, iy] = f_right((np.full_like(zn[:-1], yn[iy]), zn[:-1]))

        # Bottom boundary interpolation (ensure smooth transition)
        model[ii, -size_b:, :] = -2  # Initialize bottom boundary
        sig_bottom = np.concatenate((model[ii, -size_b - 1:-size_b, :],
                                     (-2) * np.ones_like(model[ii, -size_b - 1:-size_b, :])), 0).T
        f_bottom = RegularGridInterpolator(
            (yn[1:], np.concatenate([[zn[-size_b - 1]], [zn[int(-2 * size_b / 3)]]])),
            sig_bottom, method='linear', bounds_error=False, fill_value=None
        )
        for iz in range(-size_b, int(-2 * size_b / 3), 1):
            model[ii, iz, :] = f_bottom((yn[1:], np.full_like(yn[1:], zn[iz])))

    # Set air layer conductivity to -9 (10^-9 S/m, simulate extremely low conductivity of air)
    model[:, :nza, :] = -9
    # Convert log10(conductivity) to actual conductivity (S/m)
    model = 10 ** model 
    # Extract core region conductivity model for subsequent forward modeling
    model_k = model[:, nza:-size_b, size_b:-size_b]

    return zn, yn, freq, ry, model, zn0, yn0, model_k

zn, yn, freq, ry, sig,zn0,yn0,sig_k = generate_model(alpha_l,n_sample,n_freq,nza,size_b,size_k)

# Ensure the save directory exists
import os
save_dir = "images"
plt.rcParams['font.family'] = 'Times New Roman'
os.makedirs(save_dir, exist_ok=True)
start_inx=0
end_inx=2
# Loop to plot models from index start_inx to end_inx
for idx in range(start_inx, end_inx):
    # Create 1x2 subplot (enlarged figsize for clear visualization)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    # ---------------------- Subplot 1: Full model with extended grid ----------------------
    Y1, Z1 = np.meshgrid(yn, zn)  # Grid coordinates with extended boundaries
    h1 = ax1.pcolormesh(
        Y1 / 1e3, Z1 / 1e3, 1/sig[idx],  # Use full conductivity model
        norm=colors.LogNorm(vmin=1e0, vmax=1e4),  # Unified logarithmic normalization
        edgecolors='w', linewidth=0.5,
        cmap='jet', shading='auto'
    )
    ax1.invert_yaxis()  # Invert vertical axis (geological convention: surface at top, depth at bottom)
    ax1.set_xlabel("Distance (km)", fontsize=18)
    ax1.set_ylabel("Depth (km)", fontsize=18)
    ax1.set_title("(a)",fontsize=20)
    # ---------------------- Subplot 2: Core region only (no extended grid) ----------------------
    Y2, Z2 = np.meshgrid(yn0, zn0)  # Core region grid coordinates only
    sig_k[idx][0,...]=sig_k[idx][1,...]
    h2 = ax2.pcolormesh(
        Y2 / 1e3, Z2 / 1e3, 1/sig_k[idx],  # Use core region conductivity model
        norm=colors.LogNorm(vmin=1e0, vmax=1e4),  # Consistent color scale with subplot 1
        cmap='jet', shading='auto'
    )
    ax2.invert_yaxis()
    ax2.set_xlabel("Distance (km)", fontsize=18)
    ax2.set_ylabel("Depth (km)", fontsize=18)
    ax2.set_title("(b)",fontsize=20)

    # ---------------------- Unified color bar (concise visualization) ----------------------
    cbar = fig.colorbar(h1, ax=[ax1, ax2], orientation='horizontal', shrink=0.5, pad=0.12)
    cbar.set_label(
        r'$\log_{10}\,\rho\,(\Omega m)$', 
        fontsize=18
    )
    # ---------------------- Save figures ----------------------
    save_path1 = os.path.join(save_dir, f"model_compare_{idx:03d}.png")
    save_path2 = os.path.join(save_dir, f"model_compare_{idx:03d}.eps")
    plt.savefig(save_path1, dpi=600, bbox_inches='tight')
    plt.savefig(save_path2, dpi=600, bbox_inches='tight')
    plt.close(fig)  # Close figure to free memory

print(f"✅ Model images from {start_inx} to {end_inx-1} saved to {save_dir} folder")

✅ Model images from 0 to 1 saved to images folder


In [4]:
k = int(n_sample/num_cpus)
n_sample = int(k*num_cpus)
n_freq = np.size(freq)
n_ry   = len(ry)

# Initialize MT response arrays
rhoxy = np.zeros((n_sample,n_freq,n_ry),dtype=np_dtype) 
phsxy = np.zeros((n_sample,n_freq,n_ry),dtype=np_dtype)
rhoyx = np.zeros((n_sample,n_freq,n_ry),dtype=np_dtype)
phsyx = np.zeros((n_sample,n_freq,n_ry),dtype=np_dtype)

# Initialize electromagnetic field component arrays
ex_te = np.zeros((n_sample, n_freq, len(zn),len(yn)), dtype=np.complex64)
hx_tm = np.zeros((n_sample, n_freq, len(zn)-nza,len(yn)), dtype=np.complex64)
grad_te = np.zeros((n_sample, n_freq, len(zn)-1, len(yn)-1), dtype=np.complex64)
grad_tm = np.zeros((n_sample, n_freq, len(zn)-nza-1, len(yn)-1), dtype=np.complex64)

time0 = default_timer()
# Parallel forward modeling calculation
for ii in range(k):
    rhoxy[num_cpus*ii:num_cpus*(ii+1),:,:], phsxy[num_cpus*ii:num_cpus*(ii+1),:,:],\
        rhoyx[num_cpus*ii:num_cpus*(ii+1),:,:],phsyx[num_cpus*ii:num_cpus*(ii+1),:,:],\
        ex_te[num_cpus*ii:num_cpus*(ii+1),...],hx_tm[num_cpus*ii:num_cpus*(ii+1),...],\
        grad_te[num_cpus*ii:num_cpus*(ii+1),...],grad_tm[num_cpus*ii:num_cpus*(ii+1),...] = \
        func_remote(nza, zn, yn, freq, ry, sig[num_cpus*ii:num_cpus*(ii+1),:,:],n_sample=num_cpus,mode="TETM",use_parallel=False,num_cpus=num_cpus,num_gpus=num_gpus,ray=ray)
    print(f"{ii} of {k} finished!") 
time1 = default_timer()
print(f"Total forward modeling time for {n_sample} samples: {time1 - time0:.2f} seconds")



0 of 10 finished!
1 of 10 finished!
2 of 10 finished!
3 of 10 finished!
4 of 10 finished!
5 of 10 finished!
6 of 10 finished!
7 of 10 finished!
8 of 10 finished!
9 of 10 finished!
Total forward modeling time for 10 samples: 11.93 seconds


In [5]:
# Save complex data by splitting real and imaginary parts
ex_te1 = np.zeros((n_sample, n_freq, len(zn),len(yn), 2),dtype=np_dtype)
hx_tm1 = np.zeros((n_sample, n_freq, len(zn)-nza,len(yn),2), dtype=np_dtype)
grad_te1 = np.zeros((n_sample, n_freq, len(zn)-1, len(yn)-1),dtype=np_dtype)
grad_tm1 = np.zeros((n_sample, n_freq, len(zn)-nza-1, len(yn)-1),dtype=np_dtype)

ex_te1[:,:,:,:,0] = ex_te.real
ex_te1[:,:,:,:,1] = ex_te.imag
hx_tm1[:,:,:,:,0] = hx_tm.real
hx_tm1[:,:,:,:,1] = hx_tm.imag

grad_te1[:,:,:,:] = grad_te.real.astype(np_dtype)
grad_tm1[:,:,:,:] = grad_tm.real.astype(np_dtype)

# Define clipping limits 
clip_min = 1e-5
clip_max = 1e5

# Clip extreme values for grad_te1 (handles inf and out-of-range values)
if grad_te1 is not None:
    grad_te1 = np.clip(grad_te1, a_min=clip_min, a_max=clip_max)

# Clip extreme values for grad_tm1
if grad_tm1 is not None:
    grad_tm1 = np.clip(grad_tm1, a_min=clip_min, a_max=clip_max)


print(f"time using: {time1-time0}s")

# Save dataset with timestamp
import datetime
current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f'./datasets/test_with_grid_84_{n_sample}_{current_time}.npz'
np.savez(file_name,
         sig=sig, zn=zn, yn=yn, ry=ry, freq=freq, nza=nza,
         zn0=zn0, yn0=yn0,
         ex_te=ex_te1, hx_tm=hx_tm1,
         grad_te=grad_te1, grad_tm=grad_tm1,
         rhoxy=rhoxy,rhoyx=rhoyx,phsyx=phsyx,phsxy=phsxy
         )



time using: 11.928929099929519s


d:\ProgramPath\anaconda\envs\AMT\Lib\site-packages\numpy\_core\_methods.py:115: RuntimeWarning: overflow encountered in cast
  return um.clip(a, min, max, out=out, **kwargs)
